# 레슨 07 — 그룹화, 집계, 피벗 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 로그의 grain을 먼저 이해하고, 같은 기준으로 요약한 뒤 비교하는 것이다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/07/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 두 로그 파일 불러오기

In [ ]:
sales = pd.read_csv(f"{DATA_BASE}/sales_log.csv")
activity = pd.read_csv(f"{DATA_BASE}/user_activity_log.csv")

print("sales shape:", sales.shape)
print("sales columns:", list(sales.columns))
print("sales dtypes:")
print(sales.dtypes)
print(sales.head())

print("activity shape:", activity.shape)
print("activity columns:", list(activity.columns))
print("activity dtypes:")
print(activity.dtypes)
print(activity.head())

# sales 한 행은 판매 이벤트 1건이고, activity 한 행은 사용자 활동 이벤트 1건이다.

### 왜 이 코드가 정답인지

두 로그는 모두 날짜를 갖지만 한 행의 의미가 다르다. 매출 로그 한 행은 판매 이벤트이고, 활동 로그 한 행은 사용자 활동 이벤트다. 같은 날짜가 있어도 행 단위로 바로 합치면 잘못된 결과가 나온다. 그래서 먼저 각 로그의 구조와 grain을 확인해야 한다.

**예상 핵심값**

| 로그 | 행 수 | 열 수 |
|---|---:|---:|
| sales | 900 | 6 |
| activity | 1000 | 5 |

---

## 문제 2 정답 — 날짜형 변환과 월 컬럼 만들기

In [ ]:
sales["date"] = pd.to_datetime(sales["date"])
activity["date"] = pd.to_datetime(activity["date"])

sales["month"] = sales["date"].dt.to_period("M")
activity["month"] = activity["date"].dt.to_period("M")

sales["weekday"] = sales["date"].dt.day_name()
activity["weekday"] = activity["date"].dt.day_name()

print("sales 날짜 범위:", sales["date"].min(), "~", sales["date"].max())
print("activity 날짜 범위:", activity["date"].min(), "~", activity["date"].max())
print("sales 월별 행 수:")
print(sales["month"].value_counts().sort_index())
print("activity 월별 행 수:")
print(activity["month"].value_counts().sort_index())

### 왜 이 코드가 정답인지

날짜 문자열은 날짜형으로 바꿔야 월, 요일, 기간 기준 분석을 할 수 있다. 두 로그 모두 월별 리포트로 결합할 예정이므로 같은 방식으로 `month` 열을 만든다. `weekday` 는 요일별 확장 분석을 위한 보조 열이다. 날짜 범위와 월별 행 수를 확인하면 두 로그가 같은 기간을 덮고 있는지 알 수 있다.

**지도 메모**

날짜형 변환을 하지 않아도 문자열 정렬이 일부 맞아 보일 수 있다. 하지만 월 추출, 요일 추출, 기간 계산은 날짜형에서 안정적으로 처리된다.

---

## 문제 3 정답 — 매출 로그 기본 요약

In [ ]:
total_revenue = sales["revenue"].sum()
total_units = sales["units"].sum()
avg_order_revenue = sales["revenue"].mean()

sales["unit_price"] = sales["revenue"] / sales["units"]

print("전체 매출:", f"{total_revenue:,.0f}원")
print("전체 판매수량:", f"{total_units:,.0f}개")
print("평균 주문 매출:", f"{avg_order_revenue:,.0f}원")
print("평균 단가:", f"{sales['unit_price'].mean():,.0f}원")
print("최고 주문 매출:", f"{sales['revenue'].max():,.0f}원")

### 왜 이 코드가 정답인지

전체 매출과 판매수량은 로그 전체 규모를 보여준다. 평균 주문 매출은 한 판매 이벤트당 매출 규모를 나타낸다. `unit_price` 는 행별 매출을 수량으로 나눈 값으로, 이후 카테고리별 단가 감각을 볼 때 사용할 수 있다. 최고 주문 매출은 극단적으로 큰 주문 규모를 확인하게 해 준다.

**예상 핵심값**

```text
전체 매출: 97,308,730원
전체 판매수량: 4,505개
```

---

## 문제 4 정답 — 카테고리별 매출 집계

In [ ]:
category_sales = sales.groupby("category").agg(
    rows=("revenue", "size"),
    units=("units", "sum"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
)
category_sales["avg_unit_price"] = category_sales["revenue"] / category_sales["units"]
category_sales = category_sales.sort_values("revenue", ascending=False)

print(category_sales)
print("매출 1위 카테고리:", category_sales["revenue"].idxmax())
print("수량 1위 카테고리:", category_sales["units"].idxmax())

### 왜 이 코드가 정답인지

`groupby("category")` 는 같은 카테고리의 판매 이벤트를 묶는다. `agg` 로 주문 수, 판매수량, 매출 합계, 평균 주문 매출을 한 번에 계산할 수 있다. 평균 단가는 전체 매출을 전체 수량으로 나누어 계산해야 규모가 반영된다. 매출 1위와 수량 1위는 서로 다를 수 있으므로 따로 확인한다.

**예상 핵심값**

```text
매출 1위 카테고리: gadget
수량 1위 카테고리: toy
```

---

## 문제 5 정답 — 채널과 지역별 매출 집계

In [ ]:
channel_sales = sales.groupby("channel").agg(
    rows=("revenue", "size"),
    units=("units", "sum"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
).sort_values("revenue", ascending=False)
channel_sales["revenue_ratio"] = channel_sales["revenue"] / total_revenue * 100

region_sales = sales.groupby("store_region").agg(
    rows=("revenue", "size"),
    units=("units", "sum"),
    revenue=("revenue", "sum"),
).sort_values("revenue", ascending=False)

print("채널별 매출:")
print(channel_sales)
print("지역별 매출:")
print(region_sales)
print("매출 1위 채널:", channel_sales["revenue"].idxmax())
print("매출 1위 지역:", region_sales["revenue"].idxmax())
print("평균 주문 매출 1위 채널:", channel_sales["avg_order"].idxmax())

### 왜 이 코드가 정답인지

채널별 집계는 앱, 웹, 매장 중 어디서 매출이 큰지 보여준다. 지역별 집계는 어느 지역 매장이 강한지 보여준다. 매출 비율은 각 채널이 전체 매출에서 차지하는 비중이다. 평균 주문 매출은 주문 수 규모와 별개로 주문 한 건의 크기를 보여주므로, 매출 합계와 함께 봐야 한다.

**예상 핵심값**

```text
매출 1위 채널: store
매출 1위 지역: E
평균 주문 매출 1위 채널: store
```

---

## 문제 6 정답 — 월별 매출 추이

In [ ]:
monthly_sales = sales.groupby("month").agg(
    rows=("revenue", "size"),
    units=("units", "sum"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
).sort_index()

monthly_sales["revenue_diff"] = monthly_sales["revenue"].diff()

print("월별 매출:")
print(monthly_sales)
print("매출 최고 월:", monthly_sales["revenue"].idxmax())
print("매출 최저 월:", monthly_sales["revenue"].idxmin())
print("전월 대비 증가 최대 월:", monthly_sales["revenue_diff"].idxmax())

### 왜 이 코드가 정답인지

월별 집계는 시간 흐름을 보기 위한 기본 단위다. `sort_index()` 로 월 순서를 보장한 뒤 `diff()` 를 사용하면 전월 대비 변화량을 계산할 수 있다. 매출 최고 월과 최저 월은 규모를 보여주고, 전월 대비 증가 최대 월은 변화가 가장 컸던 시점을 보여준다.

**예상 핵심값**

매출 최고 월은 `2025-01` 이고, 매출 최저 월은 `2025-05` 다.

---

## 문제 7 정답 — 피벗 테이블: 카테고리 × 채널

In [ ]:
category_channel_pivot = pd.pivot_table(
    sales,
    index="category",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

print("카테고리 x 채널 매출 피벗:")
print(category_channel_pivot)

pivot_row_sum = category_channel_pivot.sum(axis=1)
category_revenue = sales.groupby("category")["revenue"].sum()
print("피벗 행 합계 검증:", pivot_row_sum.equals(category_revenue.loc[pivot_row_sum.index]))

print("gadget-store 매출:", f"{category_channel_pivot.loc['gadget', 'store']:,.0f}원")
print("카테고리별 매출 1위 채널:")
print(category_channel_pivot.idxmax(axis=1))

### 왜 이 코드가 정답인지

`pivot_table` 은 두 범주 기준으로 값의 합계나 평균을 표 형태로 보는 데 좋다. 여기서는 행을 카테고리, 열을 채널, 값을 매출 합계로 둔다. 피벗 행 합계가 카테고리별 총매출과 같으면 피벗이 원본 집계를 잘 반영했다는 뜻이다. 특정 칸은 `.loc[행라벨, 열라벨]` 로 꺼낸다.

**예상 핵심값**

`gadget` 카테고리의 `store` 매출은 17,040,402원이다.

---

## 문제 8 정답 — 교차표: 지역 × 채널 주문 수

In [ ]:
region_channel_count = pd.crosstab(sales["store_region"], sales["channel"])
region_channel_ratio = pd.crosstab(sales["store_region"], sales["channel"], normalize="index") * 100

print("지역 x 채널 주문 수:")
print(region_channel_count)
print("지역별 채널 비율(%):")
print(region_channel_ratio.round(1))
print("교차표 합계:", int(region_channel_count.to_numpy().sum()))
print("원본 주문 수:", len(sales))
print("E 지역 주문 수 1위 채널:", region_channel_count.loc["E"].idxmax())
print("지역별 주문 수 1위 채널:")
print(region_channel_count.idxmax(axis=1))

### 왜 이 코드가 정답인지

`crosstab` 은 두 범주형 열의 조합별 빈도를 세는 데 적합하다. `normalize="index"` 를 사용하면 각 지역 안에서 채널 비율을 볼 수 있다. 교차표 전체 합계가 원본 행 수와 같으면 모든 주문이 정확히 분류된 것이다. 피벗은 값 집계, crosstab은 빈도표라는 차이를 이해하는 것이 중요하다.

**채점 포인트**

- 주문 수 교차표와 비율 교차표를 구분했는가.
- 교차표 합계가 원본 주문 수와 같은지 확인했는가.

---

## 문제 9 정답 — 활동 로그 기본 요약

In [ ]:
total_events = len(activity)
unique_users = activity["user_id"].nunique()
total_minutes = activity["minutes"].sum()
avg_minutes = activity["minutes"].mean()
events_per_user = total_events / unique_users

print("전체 활동 이벤트 수:", total_events)
print("고유 사용자 수:", unique_users)
print("전체 활동 시간:", f"{total_minutes:,}분")
print("평균 활동 시간:", f"{avg_minutes:.2f}분")
print("사용자 1명당 평균 이벤트 수:", f"{events_per_user:.2f}건")

### 왜 이 코드가 정답인지

활동 로그는 판매가 아니라 사용자 행동 이벤트를 기록한다. 전체 이벤트 수는 로그 행 수이고, 고유 사용자 수는 중복을 제거한 사용자 수다. 한 사용자가 여러 이벤트를 만들 수 있으므로 두 값은 다르다. 전체 활동 시간과 평균 활동 시간은 이벤트 규모와 깊이를 보여준다.

**예상 핵심값**

```text
전체 활동 이벤트 수: 1000
고유 사용자 수: 257
전체 활동 시간: 23,000분
```

---

## 문제 10 정답 — 액션별 활동 집계

In [ ]:
action_summary = activity.groupby("action").agg(
    events=("user_id", "size"),
    users=("user_id", "nunique"),
    minutes=("minutes", "sum"),
    avg_minutes=("minutes", "mean"),
).sort_values("events", ascending=False)

print("액션별 활동 요약:")
print(action_summary)
print("이벤트 수 1위 액션:", action_summary["events"].idxmax())
print("총 활동 시간 1위 액션:", action_summary["minutes"].idxmax())

### 왜 이 코드가 정답인지

액션별 집계는 사용자가 어떤 행동을 많이 했는지 보여준다. `size` 는 이벤트 수, `nunique` 는 고유 사용자 수, `sum` 은 총 활동 시간, `mean` 은 이벤트당 평균 시간을 계산한다. 이벤트 수 1위와 총 활동 시간 1위가 다를 수 있으므로 둘을 분리해 본다.

**예상 핵심값**

```text
이벤트 수 1위 액션: review
총 활동 시간 1위 액션: review
```

---

## 문제 11 정답 — 사용자 등급별 활동 집계

In [ ]:
grade_activity = activity.groupby("user_grade").agg(
    events=("user_id", "size"),
    users=("user_id", "nunique"),
    minutes=("minutes", "sum"),
    avg_minutes=("minutes", "mean"),
).sort_values("events", ascending=False)

print("사용자 등급별 활동 요약:")
print(grade_activity)
print("이벤트 수 1위 등급:", grade_activity["events"].idxmax())
print("총 활동 시간 1위 등급:", grade_activity["minutes"].idxmax())

### 왜 이 코드가 정답인지

사용자 등급별 집계는 신규, 일반, VIP 사용자의 활동 규모를 비교한다. 이벤트 수가 많으면 해당 등급에서 활동 로그가 많이 발생했다는 뜻이고, 고유 사용자 수는 참여한 사용자 규모를 뜻한다. 총 활동 시간은 이벤트 수와 이벤트 길이를 함께 반영한다.

**예상 핵심값**

VIP 등급은 이벤트 수와 총 활동 시간이 가장 높게 나온다.

---

## 문제 12 정답 — 활동 피벗: 등급 × 액션

In [ ]:
grade_action_pivot = pd.pivot_table(
    activity,
    index="user_grade",
    columns="action",
    values="minutes",
    aggfunc="sum",
    fill_value=0,
)

print("등급 x 액션 활동 시간 피벗:")
print(grade_action_pivot)

pivot_minutes = grade_action_pivot.sum(axis=1)
grade_minutes = activity.groupby("user_grade")["minutes"].sum()
print("피벗 행 합계 검증:", pivot_minutes.equals(grade_minutes.loc[pivot_minutes.index]))

print("vip 등급 최장 활동 액션:", grade_action_pivot.loc["vip"].idxmax())
print("등급별 최장 활동 액션:")
print(grade_action_pivot.idxmax(axis=1))

### 왜 이 코드가 정답인지

피벗 테이블을 사용하면 등급과 액션이라는 두 기준을 동시에 볼 수 있다. 값은 활동 시간 합계로 두었으므로, 각 등급이 어떤 행동에 시간을 많이 쓰는지 알 수 있다. 피벗 행 합계가 등급별 총 활동 시간과 같으면 피벗이 원본 집계를 잘 반영한 것이다.

**예상 핵심값**

VIP 등급에서 활동 시간이 가장 긴 액션은 `purchase` 다.

---

## 문제 13 정답 — 사용자별 활동 요약

In [ ]:
user_activity = activity.groupby("user_id").agg(
    events=("action", "size"),
    total_minutes=("minutes", "sum"),
    avg_minutes=("minutes", "mean"),
    first_date=("date", "min"),
    last_date=("date", "max"),
).sort_values("events", ascending=False)

print("사용자별 이벤트 수 상위 10명:")
print(user_activity.head(10))
print("사용자별 총 활동 시간 상위 10명:")
print(user_activity.sort_values("total_minutes", ascending=False).head(10))

### 왜 이 코드가 정답인지

활동 로그는 사용자 한 명이 여러 행을 가질 수 있다. 사용자별로 묶으면 각 사용자의 활동 빈도, 총 활동 시간, 평균 활동 시간, 첫/마지막 활동일을 볼 수 있다. 이벤트 수 상위 사용자와 총 활동 시간 상위 사용자는 다를 수 있으므로 두 기준을 따로 정렬한다.

**채점 포인트**

- `user_id` 기준으로 groupby 했는가.
- 이벤트 수와 총 활동 시간을 구분했는가.
- 상위 10명을 두 기준으로 각각 출력했는가.

---

## 문제 14 정답 — 월별 매출과 활동 결합

In [ ]:
monthly_sales_report = sales.groupby("month", as_index=False).agg(
    revenue=("revenue", "sum"),
    units=("units", "sum"),
)

monthly_activity_report = activity.groupby("month", as_index=False).agg(
    events=("user_id", "size"),
    minutes=("minutes", "sum"),
    users=("user_id", "nunique"),
)

monthly_report = monthly_sales_report.merge(monthly_activity_report, on="month", how="inner")
monthly_report["revenue_per_minute"] = monthly_report["revenue"] / monthly_report["minutes"]
monthly_report["revenue_per_event"] = monthly_report["revenue"] / monthly_report["events"]

print("월별 매출+활동 리포트:")
print(monthly_report)

### 왜 이 코드가 정답인지

매출 로그와 활동 로그는 행 단위 의미가 다르므로 직접 merge 하면 안 된다. 먼저 각각 월별로 요약한 뒤 `month` 기준으로 병합해야 한다. 이렇게 하면 월별 매출, 판매수량, 활동 이벤트, 활동 시간, 사용자 수를 한 표에서 비교할 수 있다. `revenue_per_minute` 와 `revenue_per_event` 는 활동 대비 매출 효율을 보는 지표다.

**예상 핵심값**

`2025-01` 은 매출과 `revenue_per_minute` 가 모두 높게 나올 수 있다.

---

## 문제 15 정답 — 그룹화·피벗 분석 결론

In [ ]:
best_revenue_month = monthly_report.loc[monthly_report["revenue"].idxmax(), "month"]
best_minutes_month = monthly_report.loc[monthly_report["minutes"].idxmax(), "month"]
best_efficiency_month = monthly_report.loc[monthly_report["revenue_per_minute"].idxmax(), "month"]
best_category = category_sales["revenue"].idxmax()
best_action = action_summary["events"].idxmax()

print("매출 최고 월:", best_revenue_month)
print("활동 시간 최고 월:", best_minutes_month)
print("매출/활동시간 효율 최고 월:", best_efficiency_month)
print("매출 1위 카테고리:", best_category)
print("이벤트 수 1위 액션:", best_action)
print(monthly_report)

### 왜 이 코드가 정답인지

마지막 문제는 여러 집계 결과를 운영 결론으로 연결한다. 매출 최고 월은 규모를, 활동 시간 최고 월은 사용자 활동량을, 매출/활동시간 효율 최고 월은 활동 대비 매출 성과를 보여준다. 카테고리와 액션 1위까지 함께 보면 상품 성과와 사용자 행동을 함께 설명할 수 있다.

**결론 예시**

```text
매출은 2025-01이 가장 높고, 활동 시간은 2025-06이 가장 길었다.
활동 대비 매출 효율은 2025-01이 가장 높아, 같은 활동 시간 대비 매출 전환이 좋았다.
카테고리 기준으로는 gadget이 매출을 가장 크게 만들었고, 활동 액션 기준으로는 review 이벤트가 가장 많았다.
다음 분석에서는 1월의 gadget 매출과 review 활동이 어떤 채널에서 연결되는지 추가로 확인하겠다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| 구조 확인 | 1~2 | 로그 grain, 날짜형 | 두 로그를 구분하고 월 컬럼 생성 |
| 매출 집계 | 3~8 | groupby, pivot, crosstab | 카테고리/채널/지역/월 집계 |
| 활동 집계 | 9~13 | groupby, nunique, pivot | 액션/등급/사용자별 활동 요약 |
| 결합 | 14 | 같은 grain으로 병합 | 월별 요약 후 merge |
| 결론 | 15 | 규모와 효율 해석 | 숫자 기반 결론 작성 |

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| 두 로그를 바로 merge | grain 혼동 | 월별 요약 후 병합하게 함 |
| `count` 와 `nunique` 혼동 | 이벤트 수와 사용자 수 혼동 | 한 사용자가 여러 이벤트를 만들 수 있음 |
| 피벗 합계 검증 생략 | 표 오류 감지 어려움 | 행 합계와 원본 groupby 비교 |
| 평균 단가를 단순 평균으로만 계산 | 규모 반영 부족 | 보고서용은 합계 기반 계산 |
| 월별 diff 전에 정렬 안 함 | 시간 순서 오류 | 월 인덱스를 정렬한 뒤 diff |
| 결론에 효율 지표 없음 | 활동 로그 활용 부족 | 매출 규모와 활동 대비 효율을 함께 쓰게 함 |

## 부분 점수 운영 기준

1. 문제 1~2에서 날짜형 변환이 빠지면 월별 결합이 흔들리므로 먼저 수정한다.
2. 문제 3~6 매출 집계가 맞으면 매출 파트는 부분 통과 가능하다.
3. 문제 7~8 피벗과 교차표 중 하나만 맞아도 부분 인정하되, 두 도구의 차이를 질문한다.
4. 문제 9~13에서 `nunique` 를 쓰지 않으면 사용자 수 해석이 부족하므로 보완하게 한다.
5. 문제 14에서 두 로그를 원본 행 단위로 병합한 경우는 핵심 오류로 본다.
6. 문제 15 결론에 월, 카테고리, 액션 중 하나라도 빠지면 결론을 보완하게 한다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | sales와 activity의 한 행은 각각 무엇인가요? | 판매 이벤트, 사용자 활동 이벤트 |
| 2 | 두 로그에 같은 월 컬럼을 만든 이유는 무엇인가요? | 월 단위 결합 기준 |
| 3 | 매출 합계와 평균 주문 매출은 어떻게 다른가요? | 규모와 주문당 크기 |
| 4 | 카테고리별 평균 단가는 어떻게 계산했나요? | 매출 합계 / 수량 합계 |
| 5 | 채널별 매출 비율은 어떤 의미인가요? | 전체 매출 중 채널 비중 |
| 6 | 전월 대비 변화량은 왜 정렬 후 계산하나요? | 시간 순서 필요 |
| 7 | pivot_table의 행, 열, 값은 무엇인가요? | category, channel, revenue |
| 8 | crosstab은 무엇을 세나요? | 범주 조합별 빈도 |
| 9 | 이벤트 수와 고유 사용자 수는 왜 다른가요? | 한 사용자가 여러 이벤트 가능 |
| 10 | 총 활동 시간과 평균 활동 시간은 어떻게 다른가요? | 규모와 이벤트당 시간 |
| 11 | 등급별 활동을 볼 때 users가 필요한 이유는 무엇인가요? | 사용자 규모 확인 |
| 12 | 피벗 행 합계를 검증하는 이유는 무엇인가요? | 원본 집계와 일치 확인 |
| 13 | 사용자별 요약은 어떤 운영 질문에 답하나요? | 많이 활동한 사용자 확인 |
| 14 | grain이 다른 데이터를 어떻게 합쳤나요? | 월별 요약 후 병합 |
| 15 | 좋은 결론에는 어떤 축이 들어가나요? | 매출, 활동, 효율, 다음 행동 |

## 재실행 확인 순서

1. 런타임을 새로 시작한다.
2. 환경 셀부터 문제 15까지 순서대로 실행한다.
3. `sales` 와 `activity` 모두 `month` 열이 생겼는지 확인한다.
4. `monthly_report` 의 월이 2025-01부터 2025-06까지 있는지 확인한다.
5. 결론의 월과 카테고리, 액션이 코드 출력과 일치하는지 확인한다.

정답 코드와 학생 코드가 달라도 된다. 다만 groupby 기준, 피벗의 행/열/값, 월별 병합 기준을 말로 설명하지 못하면 보충 설명을 요구한다.

## 보충 설명 포인트

- 로그 데이터는 "행 하나가 무엇인가"를 먼저 확인해야 한다. 매출 로그와 활동 로그가 모두 날짜를 갖고 있어도 같은 행끼리 직접 연결할 수 있는 것은 아니다.
- `groupby` 는 긴 로그를 의사결정 가능한 요약표로 줄이는 도구다. 학생에게 "어떤 기준으로 묶었는가"와 "무엇을 합계/평균 냈는가"를 계속 말하게 한다.
- `pivot_table` 은 한 값 열을 두 범주 축으로 펼쳐 보는 도구다. 행과 열이 모두 범주이고, 셀에는 집계된 값이 들어간다는 그림을 먼저 잡아 주면 이해가 빠르다.
- `crosstab` 은 값 열이 없어도 빈도를 셀 수 있다. 이번 문제에서 지역×채널 주문 수를 보는 이유는 "매출 금액"이 아니라 "주문 발생 빈도"를 보고 싶기 때문이다.
- `nunique` 는 로그 분석에서 자주 쓰인다. 이벤트가 1000건이어도 고유 사용자는 257명일 수 있으므로, 이벤트 수와 사용자 수를 섞어 말하지 않게 한다.
- 월별 결합은 grain 조정의 예시다. 서로 다른 로그라도 같은 월 단위로 요약하면 안전하게 비교할 수 있다.

## 수업 중 피드백 문장 예시

- "지금 만든 표에서 한 행은 어떤 기준의 한 줄인가요?"
- "이 숫자는 이벤트 수인가요, 고유 사용자 수인가요?"
- "피벗 테이블의 행, 열, 값이 각각 무엇인지 먼저 말해 봅시다."
- "매출이 높은 것과 효율이 높은 것은 같은 말이 아닙니다."
- "두 로그를 합치기 전에 같은 단위로 요약했는지 확인합시다."
- "결론에는 매출 규모와 활동 지표가 둘 다 들어가야 합니다."

## 부분 점수 세부 기준

| 상황 | 처리 |
|---|---|
| 매출 로그 집계만 정확함 | 매출 파트 부분 통과, 활동 파트 보충 |
| 활동 로그 집계만 정확함 | 활동 파트 부분 통과, 월별 결합 보충 |
| 피벗은 맞지만 검증 없음 | 통과 가능하나 행 합계 검증 피드백 |
| `merge` 를 원본 행 단위로 수행 | 핵심 오류, 월별 요약 후 다시 작성 |
| 결론에 효율 지표 없음 | 코드 점수 인정, 결론 재작성 |
| 숫자는 맞지만 단위 없음 | 보고서 품질 피드백 |

## 최종 확인 질문

학생 제출물을 보기 전에 아래 질문에 답하게 하면 핵심 이해도를 빠르게 확인할 수 있다.

1. `sales` 와 `activity` 를 왜 바로 합치면 안 되는가?
2. 카테고리별 매출 1위와 수량 1위가 다를 수 있는 이유는 무엇인가?
3. `pivot_table` 과 `crosstab` 의 가장 큰 차이는 무엇인가?
4. 이벤트 수와 고유 사용자 수는 왜 다르게 봐야 하는가?
5. `revenue_per_minute` 는 어떤 운영 질문에 답하는가?

이 다섯 질문에 답하면 7강의 핵심인 groupby 기준 설정, 로그 grain 이해, 피벗/교차표 구분, 월별 결합 해석을 대체로 통과한 것으로 본다.